# Step 16.5 — Toy Sampling vs Industrial Input Pipelines

## 从随机窗口，到顺序、打包、分片与可恢复的位置

16.3 已经建立 `get_batch(data, batch_size, block_size, rng)`，把一个选定 split 变成 `X/Y:(B,T)`；16.4 已区分 seed 与 RNG state。本节把这两个小接口放进更完整的 **input pipeline（输入管线）** 中理解。

输入管线负责把数据组织成模型可用的 batches。数据量、文档数量与并行读取规模增大后，还要明确五件事：**random windows、shuffling、token packing、distributed shards、deterministic cursor/state**。它们可以组合，不代表存在一套所有工业系统都相同的实现。

本节只准备小型、可手算的机制示例；项目继续使用 Python 3.12 与 MLX，不安装数据框架，不改现有 MiniGPT，不启动分布式。所有 code cells 均未执行，无采样或实验结果，也没有练习答案。

材料状态：16.4 的完整回复、母版和 Step 16 文档已由**材料审核方**独立核验核心静态 PASS，这不是学员学习或运行反馈。累计实际修正保留 16.1=1/2、16.2=1/2、16.3=0/2、16.4=0/2；16.5 本次初版=**0/2**。

**本轮完成后停在大 Step 16，不进入 Step 17。**


## 0. 新 Work Copy 的依赖与使用方式

本节所有输入都是母版里写明的手工整数与短列表，**不需要读取 16.1 的语料缓存**，也不依赖另一个 Notebook 的 kernel state。

下面带入 `random`、`mlx.core` 导入，以及 16.3 原有 `get_batch` 定义。之后需要的 packing、rank plan、cursor helper 都在本节首次使用前定义。新副本无需 `%run` 旧 Notebook，也无需复制旧 kernel 变量。

以后使用时，先在项目 `.venv` 的 Python 3.12 kernel 中运行这个定义 cell，再按各节的“定义 → toy usage”顺序继续即可。`mx.array`、整数 dtype、二维切片与 `(B,T)` 沿用前节；新 API 会在对应代码之前解释。

本节的 toy IDs 只是结构示意，未关联真实 `stoi/itos`。它们不会被送入 MiniGPT，也不更改前节真实字符词表。若以后要回到真实语料，69/70 母版已经提供既有缓存路径与 bootstrap；本节不用重新读取它来讨论管线机制。

以下导入与函数定义、以及后续 usage 都是准备的材料，本轮不执行。


In [ ]:
import random
import mlx.core as mx


def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


# Definition only. No batch is sampled in this cell.


## 1. Random Windows：当前 Sampler 做好了什么？

当前输入是一条已经选好的整数 split：`data:(N_split,)`。每次调用随机选 B 个合法起点，每行读取 T+1 个连续 tokens：

$$
0\le s<N_{split}-T,\qquad
X[b,t]=data[s_b+t],\quad Y[b,t]=data[s_b+t+1].
$$

**Random window（随机窗口）** 的随机性在于起点。窗口内部仍按原来的文本顺序排列。

每次 `randrange` 都重新从整个合法集合选取，属于 **sampling with replacement（有放回抽样）**：起点允许重复，窗口允许重叠。因此“取了若干批”并不表示“恰好把所有候选窗口看了一遍”。16.4 的 RNG state 记录随机过程走到哪里，不记录一份已经完整遍历的窗口清单。

小例子取 `N_split=12,T=4`，合法起点为 `0…7`。若某两行的起点分别是 2、3，两行共享不少 tokens；若起点都是 2，两行可以完全相同。这只是可能情况，不是某个 seed 的实测输出。

### 只准备一个独立的 Toy Usage

`list(range(12))` 给出手工定义的整数 0 到 11；`mx.array(..., dtype=mx.int32)` 把它们组织成一维 token stream。`window_rng` 是本示例自己的 RNG，不使用 16.4 work copy 中的 train/val 对象。

当前方法很适合解释 next-token 配对，也能用于较大的数据；当需要控制整轮覆盖、顺序读取多个文件或并行分配时，还需增加相应机制，不能仅通过增大 B 解决。


In [ ]:
# USER-RUN TOY WINDOWS — prepared only, NOT executed in the master.

toy_stream = mx.array(list(range(12)), dtype=mx.int32)
window_block_size = 4
window_rng = random.Random(2026)

X_window, Y_window, window_starts = get_batch(
    toy_stream, batch_size=2, block_size=window_block_size, rng=window_rng,
)

print('window starts:', window_starts)
print('window X/Y shapes:', X_window.shape, Y_window.shape)

# Expected by construction: X/Y (2,4); starts length 2, values in 0..7.
# No particular random starts or measured outputs are supplied.


## 2. Shuffling：打乱样本顺序，保留样本内部词序

**Shuffling（重排 / 洗牌）** 可以先建立有限候选列表，再重新排列它的访问顺序。本节先把一个候选单位定义为一个合法窗口起点；工业管线也可能重排文档、已经形成的样本、或文件分片。

对刚才 8 个合法起点，可手工举出一种排列：

```text
[3,0,7,1,4,2,6,5]
```

这是为了说明结构而指定的排列，不是下面代码的随机输出。完整走过这个列表时，每个候选 ID 出现一次；每个窗口内部仍按连续 token 次序读取。不同起点对应的窗口依然可能重叠。

**Epoch（一轮遍历）** 必须先说明遍历单位和规则。本例可将“完整走过这个有限候选列表”叫一轮。若后面选择丢弃尾部或补齐重复样本，覆盖语义会随之改变；无限流也未必有自然的完整一轮。

### 首次使用前解释 `rng.shuffle`

`rng.shuffle(order)` 在**原列表上**重排元素，返回 `None`；不要写成 `order = rng.shuffle(order)`。它不会修改这些起点所指向的 `toy_stream`。

下面先用 `list(range(...))` 创建起点列表，再用独立的 `shuffle_rng` 重排。16.4 的条件继续适用：相同初始状态、同样的输入与操作顺序，才有相应的复现预期。

大数据流未必能把所有候选放进内存。**Shuffle buffer（洗牌缓冲区）** 可先容纳有限条记录，选出一条后补进后续记录；这是有限缓冲的混合方式，不能直接称为整个数据集的一次全局随机排列。这里只解释机制，不实现流式缓冲。[Python shuffle](https://docs.python.org/3.12/library/random.html#random.shuffle) · [官方 streaming shuffle 说明](https://huggingface.co/docs/datasets/en/stream#shuffle)


In [ ]:
# USER-RUN FINITE SHUFFLE — prepared only, NOT executed in the master.
# Depends on the toy_stream and window_block_size setup above.

start_order = list(range(toy_stream.shape[0] - window_block_size))
shuffle_rng = random.Random(2027)
shuffle_rng.shuffle(start_order)

print('one candidate-start order:', start_order)

# This prepares an order; it does not loop through an epoch or train a model.
# The original toy_stream and each window's internal token order stay intact.


## 3. Token Packing：怎样把不同长度的数据组成固定窗口？

**Token packing（token 打包）** 将多段数据组织成较满的序列，以减少短序列单独补齐所留下的空位。**Padding（补齐）** 是为统一形状而加入占位位置；占位位置是否参与可见上下文和监督，需要单独约定。本节不实现 padding。

先区分：**packing 沿时间轴安排 tokens；batching 沿 batch 轴把多行放在一起。** 它们不是同一个动作。

### 一个明确选择连续流语义的例子

手工定义三份 toy 文档：

```text
A = [10,11,12]
B = [20,21]
C = [30,31,32,33]
```

**EOS（End Of Sequence，序列结束标记）** 是一种专门的 token。本例约定 toy ID `99` 表示文档末尾，在每份文档后加入一次，得到：

```text
[10,11,12,99,20,21,99,30,31,32,33,99]
```

长度 L=12。这里的 `99` 只属于这个手工示意，**不是现有 Tiny Shakespeare 字符词表中的新增 ID**；当前 tokenizer 和 MiniGPT 均不修改。

本例选择：把上述内容当成一条连续流，允许跨文档的 next-token 配对。令 T=4，每行仍需 T+1=5 个 tokens。为了连续覆盖相邻配对，下一行起点前进 T，而不是 T+1：

| 行 | 起点 | T+1 chunk | X | Y |
|---|---:|---|---|---|
| 0 | 0 | `[10,11,12,99,20]` | `[10,11,12,99]` | `[11,12,99,20]` |
| 1 | 4 | `[20,21,99,30,31]` | `[20,21,99,30]` | `[21,99,30,31]` |

这两行组成 batch 后，`X/Y:(2,4)`。前一行最后的 target 20 也是后一行第一个 input；它们在不同监督位置承担不同角色，没有重复同一个相邻 token 配对。

### 数学与尾部策略

对长度 L>=1 的流，完整行数是：

$$
M=\left\lfloor\frac{L-1}{T}\right\rfloor.
$$

减去 1，是因为需要为输入留下一个后继 target。第 i 行从 `i*T` 开始。已形成 M 行后，保留：

```python
carry = stream[M*T:]
```

**Carry / remainder（待继续装填的余量）** 包含下一行的起始 token。本例返回 `[31,32,33,99]`：31 已作为上一行最后的 target，接下来可作为新行第一个 input；其余还包含 3 对尚未形成完整输出行的相邻配对。还需后续 token 才能凑足下一行的 T+1 个 tokens。

若后续仍有数据，应把 carry 接在新到的 tokens 前继续处理，不能再次插入已经存在的 EOS。若数据到此结束，必须明确选择尾部舍弃、较短行或带规则的 padding。本 helper 只把 carry 返回，不默默丢弃、补齐或绕回开头。

### EOS、Attention Boundary 与 Target Boundary

两类规则必须分别决定：

- **Attention boundary（注意力可见边界）**：后一个文档的输入能否看见同一行里较早文档的内容。
- **Target boundary（监督配对边界）**：是否保留 `EOS → 下一文档首 token` 这样的配对。

本例两者都按连续流处理；例如 `99→20` 是被保留的 target 配对，第二行普通 causal attention 可让 C 的 token 30 看见该行里 B 的前文。**插入 EOS 本身不会自动建立隔离 mask，也不会自动取消跨文档 target。**

若目标是把多个文档放进同一行却仍保持文档间隔离，就要记录边界，并让 attention 和目标处理真正使用相应规则。当前 MiniGPT 没有接入这种文档边界接口；本节只解释区别，不改模型、不实现 loss。[官方实现中的 attention / target 分离配置](https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/datasets/gpt_dataset.py)

所有 packing、shuffle 和后续分片都应在已经选定的 split 内进行，不能重新把 validation 数据混进 training。


### Packing Helper：先解释使用到的列表操作

`stream.extend(document)` 按原顺序加入该文档的每个 token；`append(99)` 只加入一个结束标记。整数 `//` 做向下取整的除法；`max(0, ...)` 让空输入也得到零行。`range(0, stop, T)` 依次给出 0、T、2T 等起点，不包含 stop。

下面的 helper 只接收已经组织好的一条 Python token list，不决定 tokenizer、文档顺序或 EOS 策略。它返回等长的 chunk 列表和 carry；若没有完整行，就返回空 chunk 列表，调用方应先等到有完整行再组成 batch。


In [ ]:
def pack_token_stream(stream, block_size: int):
    """Prepare length-(T+1) chunks at stride T, retaining the next input tail."""
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    full_rows = max(0, (len(stream) - 1) // block_size)
    chunks = [
        stream[start : start + block_size + 1]
        for start in range(0, full_rows * block_size, block_size)
    ]
    carry = stream[full_rows * block_size :]
    return chunks, carry


# Definition only. No documents are packed by this cell.


In [ ]:
# USER-RUN TOY PACKING — prepared only, NOT executed in the master.

toy_documents = [[10, 11, 12], [20, 21], [30, 31, 32, 33]]
toy_eos_id = 99  # Illustration only; not added to the real character vocabulary.
pack_block_size = 4

packed_stream = []
for document in toy_documents:
    packed_stream.extend(document)
    packed_stream.append(toy_eos_id)

packed_chunks, packing_carry = pack_token_stream(
    packed_stream, pack_block_size,
)

# These explicitly chosen toy inputs provide two complete equal-length rows.
# mx.array turns the rectangular nested list into shape (2,5).
packed_rows = mx.array(packed_chunks, dtype=mx.int32)
X_packed = packed_rows[:, :-1]
Y_packed = packed_rows[:, 1:]

print('packed X/Y shapes:', X_packed.shape, Y_packed.shape)
print('tokens retained for continuation:', packing_carry)

# By construction: X/Y (2,4), carry [31,32,33,99]. Not measured here.
# No real corpus, real vocabulary, model, padding, or loss is used.


## 4. Distributed Shards：先明确数据放在哪里、由谁处理

数据变大后，存储和处理可能被拆开。先区分四个词：

| 术语 | 本节含义 |
|---|---|
| **Storage shard（存储分片）** | 数据的一部分，例如一个文件或文件内可独立读取的块 |
| **Rank** | 一组协作进程中的编号，取值 `0…R-1` |
| **World size = R** | 这组进程的数量；它不是固定意义上的 GPU 数量 |
| **Loader worker** | 帮某个进程读取或处理数据的工作单元，一个 rank 可以有多个 |

一个 shard 可以包含很多文档/样本；shard 数与 rank 数不必相同。**Train/validation split** 决定数据的用途隔离；**sharding / rank assignment** 决定存储或工作分配，不能互相代替。

下面只在一个 Notebook 中写出各 rank 的样本分配计划。不会启动进程、建立通信组或安装分布式数据框架。

### 从同一个全局 Order 分配

换一个独立的手算场景：现在有 10 条已经形成的样本，ID 为 `0…9`。这里的 **sample ID 是记录编号，不是 token ID，也不是前面那条流的起点**。它指向一条完整的 T+1 token chunk。

先手工指定所有 rank 共同认可的顺序：

```text
order = [7,2,9,4,0,6,1,8,3,5]
```

这不是实际 shuffle 输出。实际使用时可以共享已保存的 order，或在相同数据/算法/seed/epoch 条件下重建它。先各自用不同 seed 打乱全集，再各取一部分，不能保证彼此没有重复或遗漏。

令 R=2，每个 rank 的 **local batch size** 为 $B_{local}=2$。一次逻辑全局 batch 含：

$$
B_{global}=R B_{local}=4.
$$

本例采用 **drop-global-tail policy（丢弃不足一整个全局 batch 的尾部）**。若样本总数为 M，可使用的记录数：

$$
M_{usable}=\left\lfloor\frac{M}{R B_{local}}\right\rfloor(R B_{local})=8.
$$

| 逻辑 batch 编号 | Rank 0 | Rank 1 |
|---|---|---|
| 0 | `[7,2]` | `[9,4]` |
| 1 | `[0,6]` | `[1,8]` |

尾部 `[3,5]` 在本轮计划中明确不使用。两个 rank 各有两份完整 local batches；本例没有声称覆盖全部 10 条。其他策略可以选择保留不等长尾部、补齐或重复样本，但要说明由此改变的步数和覆盖语义。

若 T=4，每个 rank 的 `X/Y:(2,4)`；合计逻辑 batch 有 4 行，可记作 `(4,4)`。不需要真的把两个 rank 的输入集中在一台机器上。这里仅讨论每个 rank 一份 local batch 的数据分配，不涉及梯度累积。

当 order 中 sample IDs 唯一时，此方案的已保留位置不会跨 rank 重复分配；样本的文本内容仍可能重复或重叠。所有 IDs 都来自同一个已选 split。[官方共享 shuffle 顺序的分布式说明](https://huggingface.co/docs/datasets/en/use_with_pytorch#distributed)


### 分配 Helper：每个全局 Batch 内给各 Rank 连续一段

`offset` 是当前全局 batch 在 order 中的起点；`offset + rank * local_batch_size` 是这个 rank 的局部段起点。按固定段长切片，就与上表对应。

前置约定是 order 包含唯一的整数 sample IDs。`set(order)` 在这里仅检查是否存在重复 ID，不改变 order。函数显式检查 rank 与数量参数，只返回 Python 列表计划，不读取任何样本。

若样本总数不足一个全局 batch，返回空计划且全部进入 dropped tail；这是本例所选策略的直接结果。


In [ ]:
def make_rank_plan(order, rank: int, world_size: int, local_batch_size: int):
    """Partition one shared sample order, dropping only its incomplete global tail."""
    if type(world_size) is not int or world_size < 1:
        raise ValueError('world_size must be a positive Python int')
    if type(local_batch_size) is not int or local_batch_size < 1:
        raise ValueError('local_batch_size must be a positive Python int')
    if type(rank) is not int or not 0 <= rank < world_size:
        raise ValueError('rank must satisfy 0 <= rank < world_size')
    if len(set(order)) != len(order):
        raise ValueError('this teaching plan requires unique sample IDs')

    global_batch_size = world_size * local_batch_size
    usable = (len(order) // global_batch_size) * global_batch_size
    local_batches = []
    for offset in range(0, usable, global_batch_size):
        start = offset + rank * local_batch_size
        local_batches.append(order[start : start + local_batch_size])

    return local_batches, order[usable:]


# Definition only. No distributed process or sample access is created.


### 从 Sample IDs 到 `(B,T)` 的连接

为让 shape 有具体依托，再定义 10 条短 toy records：第 i 条存放 `[10*i, ..., 10*i+T]`，每条都是 T+1 个 token IDs。例如 sample 7 的 chunk 为 `[70,71,72,73,74]`。它们与前面的 packing 文档是两个独立例子，不共享词表，也不送进模型。

用一批 sample IDs 索引这些完整 chunks，得到 `(B_local,T+1)`；再像16.3一样分别去掉末项和首项，就得到 X/Y。下面只预备展示 rank 0 的第一份 local batch；所有数字都由手工输入定义，不是采样结果。


In [ ]:
# USER-RUN RANK PLAN — prepared only, NOT executed in the master.

manual_order = [7, 2, 9, 4, 0, 6, 1, 8, 3, 5]
world_size = 2
local_batch_size = 2
plan_block_size = 4

rank0_plan, dropped_tail = make_rank_plan(
    manual_order, 0, world_size, local_batch_size,
)
rank1_plan, _ = make_rank_plan(
    manual_order, 1, world_size, local_batch_size,
)

toy_records = [
    list(range(10 * sample_id, 10 * sample_id + plan_block_size + 1))
    for sample_id in range(10)
]
preview_ids = rank0_plan[0]
local_rows = mx.array([toy_records[i] for i in preview_ids], dtype=mx.int32)
X_local = local_rows[:, :-1]
Y_local = local_rows[:, 1:]

print('rank 0 plan:', rank0_plan)
print('rank 1 plan:', rank1_plan)
print('dropped global tail:', dropped_tail)
print('rank 0 first X/Y shapes:', X_local.shape, Y_local.shape)

# By construction: rank0 [[7,2],[0,6]], rank1 [[9,4],[1,8]], tail [3,5].
# This chosen example gives X/Y (2,4); no communication or training occurs.


## 5. Deterministic Cursor / State：下一次到底从哪里取？

**Cursor（游标）** 是某个明确顺序里的当前位置。本节把 `next_batch_index` 定义为：**这份 rank-local plan 中下一批的编号**。它不是 sample ID、token 下标或已完成的 training update 数。

例如 rank 0 的计划是 `[[7,2],[0,6]]`：

| `next_batch_index` | 含义 |
|---:|---|
| 0 | 下一批是 `[7,2]` |
| 1 | 下一批是 `[0,6]` |
| 2 | 本轮这个 rank 的计划已到结尾 |

单独保存 `1` 没有充分意义：必须知道它属于哪份 order、哪个 epoch、哪个 rank，以及怎样分配和处理尾部。改变 world size 或 local batch size 后，同样的本地 cursor 不再自动对应原来的样本。

### 小型内存计划：Order + Cursor

本例的整份 order 已经在内存里固定，没有在线 shuffle、随机变换或预取。保存这个 order、分配配置和 cursor，即可定位当前 epoch 的下一批 IDs。**此时定位下一项不再需要抽随机数**；如果之后要继续生成新顺序或执行其他随机操作，仍需16.4所讲的 RNG state 或可重建配置。

`tuple(order)` 把整数 ID 顺序复制为不可变的 tuple，避免原 list 后来被重排而影响快照。`dict` 保存各字段。`list.copy()` 给返回的一批 ID 创建独立列表。

下方 helper 返回一批 IDs 和新的 cursor；本 toy 将一次成功返回视为已交付给它的消费者，不代表模型已经训练这批数据。到结尾时返回 `None`，不自动绕回第0批或开始新 epoch。


In [ ]:
def take_planned_ids(local_plan, next_batch_index: int):
    """Return the next planned IDs and the cursor after this toy delivery."""
    if type(next_batch_index) is not int or not 0 <= next_batch_index <= len(local_plan):
        raise ValueError('cursor must point to a batch or the end of this plan')
    if next_batch_index == len(local_plan):
        return None, next_batch_index
    return local_plan[next_batch_index].copy(), next_batch_index + 1


# Definition only. No plan is consumed in this cell.


In [ ]:
# USER-RUN PLAN SNAPSHOT — prepared only, NOT executed in the master.
# Requires the unchanged manual_order, rank0_plan, and toy_records above.

next_batch_index = 0
delivered_ids, next_batch_index = take_planned_ids(rank0_plan, next_batch_index)

plan_snapshot = {
    'dataset_id': 'toy_records_v1',  # A manual label, not a content checksum.
    'epoch': 0,
    'order': tuple(manual_order),
    'rank': 0,
    'world_size': world_size,
    'local_batch_size': local_batch_size,
    'block_size': plan_block_size,
    'tail_policy': 'drop_global_tail',
    'next_batch_index': next_batch_index,
}

# With these chosen inputs, the saved cursor is after delivery of [7,2].
# This is an in-memory plan snapshot, not a training checkpoint or file.


### 恢复的是这份计划的位置

以下代码要求同一份 `toy_records`、同一套 helper 和软件环境仍在当前 kernel 中。`dataset_id` 只是人工标记，不能证明内容未变；真实系统还需可核对的数据版本与内容身份。

恢复时使用快照的 order、rank、world size、batch size 和相同尾部策略重建计划，再从保存的 cursor 取下一批。本例选择的输入使下一批为 `[0,6]`，这是对手工计划的推导，不是一次运行验证。`block_size` 也从快照恢复，用于解释样本 shape；数据必须仍按这一配置准备。

这个快照只演示固定计划，不包含前面的 packing carry，也不代表一个完整流式 loader。Kernel 重启后内存字典消失，持久化和训练恢复仍留给 Step 20。


In [ ]:
# USER-RUN IN-MEMORY PLAN RESTORE — prepared only, NOT executed in the master.

if plan_snapshot['tail_policy'] != 'drop_global_tail':
    raise ValueError('unsupported tail policy for this teaching helper')

restored_plan, restored_tail = make_rank_plan(
    list(plan_snapshot['order']),
    plan_snapshot['rank'],
    plan_snapshot['world_size'],
    plan_snapshot['local_batch_size'],
)
restored_block_size = plan_snapshot['block_size']
restored_cursor = plan_snapshot['next_batch_index']
next_ids, restored_cursor = take_planned_ids(restored_plan, restored_cursor)

print('next planned sample IDs:', next_ids)
print('cursor after toy delivery:', restored_cursor)
print('context length for unchanged toy records:', restored_block_size)

# Re-running this restore cell returns to the same saved boundary.
# To continue later, retain the updated cursor rather than restoring again.


### 工业管线的 State 往往比一个 Cursor 更大

16.4 的随机窗口 sampler 主要保存 RNG state；本节的固定计划还需要 order 与 cursor。若又加入流式读取、有限缓冲与在线 packing，下一输出还取决于更多对象。

| 状态 / 配置 | 为什么会影响下一份输出 |
|---|---|
| 数据版本、词表、split、shard 清单及顺序 | 相同位置必须仍指向相同数据，保持 train/validation 隔离 |
| epoch、样本 order 或它的可重建规则 | 说明正在遍历哪一轮、哪一份顺序 |
| rank / world size / worker 分配与尾部策略 | 说明该消费者负责哪部分以及遗漏或补齐规则 |
| reader 当前 shard 与 shard 内位置 | 确认接下来从哪里读取 |
| RNG state | 继续尚未发生的随机操作 |
| shuffle buffer 中已有但尚未输出的记录 | 相同 RNG state 配不同缓冲内容，会得到不同样本 |
| packing carry 与相关文档边界信息 | 继续构造尚未完成的序列，而不丢失 tokens 或误改边界 |
| 已消费位置及必要的 pending queues / 重建规则 | 防止跳过或重复已经读出、但尚未被下游处理的数据 |

**Prefetch（预取）** 是提前读取后面的数据，以减少消费者等待。假如 reader 已读到第12项，消费者只确认用到第8项，9–12仍待处理。只保存 reader 的位置，然后从13恢复，就可能跳过它们。

恢复协议可以保存待处理内容，也可以从明确的已消费边界确定性重建；关键是说明采取哪一种。不能把“提供一个 state API”自动理解成“在所有配置下精确重放”。例如 Hugging Face 的 streaming 文档明确记录了 shuffle-buffer 恢复的限制；MLX Data 也区分有序与非确定顺序的预取路径。这里仅用官方文档说明机制，不引入相应依赖或 API 实现。

本节没有实际 reader、后台队列或在线 packing 引擎，表格是理解工业管线需要哪些状态的范围说明。完整训练还涉及模型和 optimizer 等状态，沿用16.4的清单，本轮不进入 Step17/20 的实现。

[Streaming 状态与恢复边界](https://huggingface.co/docs/datasets/en/stream#save-a-dataset-checkpoint-and-resume-iteration) · [MLX Data 的 buffer、stream 与预取](https://ml-explore.github.io/mlx-data/build/html/buffers_streams_samples.html)


## 五种机制怎样对应回本项目？

| 机制 | 主要决定的问题 | 本节保留的接口 / 约束 |
|---|---|---|
| Random windows | 从当前 split 哪个起点取窗口？ | 每行 T+1，输出 X/Y `(B,T)`，可重复 |
| Shuffling | 候选记录按什么次序被访问？ | 重排 IDs，不破坏窗口内 token 次序 |
| Token packing | 短文档/连续流怎样形成有效序列？ | 明确文档边界、shift、carry 与尾部策略 |
| Distributed shards / assignment | 数据如何存储、由哪个消费者负责？ | shared order、rank 分配、local/global batch 与尾部一致 |
| Deterministic cursor/state | 下一次应从什么状态继续？ | 数据身份 + 顺序 + 位置 + 实际存在的随机/缓冲状态 |

Random windows 也可以放在工业管线内；工业系统不一定全部采用同一个 shuffle、packing 或分片顺序。本节的示例只把职责拆清楚，不将简化计划包装为生产级实现。

当前课程继续保留16.1–16.4已经通过的字符编码、split和sampler。Toy EOS、示例records与rank plan都留在本节说明中，不改变真实数据或模型输入协议。


## Work Copy 的可操作顺序与大 Step 16 的终点

之后实际使用时：

1. 在仓库根目录更新，再以新文件名和 `cp -n` 复制本节母版；只打开 work copy，选择项目 Python 3.12 / `.venv` kernel。
2. 运行 imports 与原 sampler 定义。无需读取 corpus cache，也无需共享69/70的变量。
3. 如需观察随机窗口与shuffle，再依次使用对应 toy setup / usage；随机结果不预填。
4. 阅读文档边界与shift规则，定义 `pack_token_stream`，再使用自带的三份toy文档。
5. 阅读rank与尾部策略，定义 `make_rank_plan`，再使用明确给出的手工order和toy records。
6. 定义 `take_planned_ids`，再按“交付一批 → 内存快照 → 重建同一计划并定位下一批”的顺序使用。固定计划示例不运行模型，也不宣称训练已消费数据。

这些是未来work副本的操作说明，本轮没有执行任何Notebook、测试、实验、采样、训练或用户副本，没有读取/下载语料，也没有代答练习。

至此 **Step16.1–16.5 的课程材料已全部准备**，16.5仍是本次初版，不能将其写成全部独立验收、学员学完或运行验证。

**本轮停在大 Step16，不进入Step17。** 累计实际修正为16.1=1/2、16.2=1/2、16.3=0/2、16.4=0/2、16.5初版=0/2，跨轮与换对话不重置。后续只按请求逐小节推进，至大Step20全部完成后停止。


## 参考与相邻课程

- [16.3 Batch sampler](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/69_step16_3_batch_sampler.ipynb) 与 [16.4 Reproducibility](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/70_step16_4_reproducibility.ipynb)：保持已通过的接口和教学内容。
- [Python 3.12 random.shuffle](https://docs.python.org/3.12/library/random.html#random.shuffle)：原地重排API。
- [Hugging Face Datasets streaming](https://huggingface.co/docs/datasets/en/stream)：有限shuffle buffer、shards和恢复保证的实例。
- [Hugging Face Datasets distributed说明](https://huggingface.co/docs/datasets/en/use_with_pytorch#distributed)：共享shuffle顺序与rank分配的关系；本课没有引入该文的框架依赖。
- [NVIDIA Megatron-LM GPT dataset](https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/datasets/gpt_dataset.py)：attention边界和目标处理分别配置的官方实现依据。
- [NeMo 24.07 sequence packing](https://docs.nvidia.com/nemo-framework/user-guide/24.07/nemotoolkit/features/optimizations/sequence_packing.html)：连接序列与隔离文档的不同packing语义，仅作为机制参考，不照搬其框架或训练配置。
- [MLX Data Buffers, Streams and Samples](https://ml-explore.github.io/mlx-data/build/html/buffers_streams_samples.html)：数据访问与预取次序的官方背景；本节不安装mlx-data。
- [Step16文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-16-tiny-shakespeare-data.md) 与 [roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md)。项目继续固定 `mlx==0.32.2`。
